# **New York City Yellow Taxi Data**

## Objective
In this case study you will be learning exploratory data analysis (EDA) with the help of a dataset on yellow taxi rides in New York City. This will enable you to understand why EDA is an important step in the process of data science and machine learning.

## **Problem Statement**
As an analyst at an upcoming taxi operation in NYC, you are tasked to use the 2023 taxi trip data to uncover insights that could help optimise taxi operations. The goal is to analyse patterns in the data that can inform strategic decisions to improve service efficiency, maximise revenue, and enhance passenger experience.

## Tasks
You need to perform the following steps for successfully completing this assignment:
1. Data Loading
2. Data Cleaning
3. Exploratory Analysis: Bivariate and Multivariate
4. Creating Visualisations to Support the Analysis
5. Deriving Insights and Stating Conclusions

---

**NOTE:** The marks given along with headings and sub-headings are cumulative marks for those particular headings/sub-headings.<br>

The actual marks for each task are specified within the tasks themselves.

For example, marks given with heading *2* or sub-heading *2.1* are the cumulative marks, for your reference only. <br>

The marks you will receive for completing tasks are given with the tasks.

Suppose the marks for two tasks are: 3 marks for 2.1.1 and 2 marks for 3.2.2, or
* 2.1.1 [3 marks]
* 3.2.2 [2 marks]

then, you will earn 3 marks for completing task 2.1.1 and 2 marks for completing task 3.2.2.


---

## Data Understanding
The yellow taxi trip records include fields capturing pick-up and drop-off dates/times, pick-up and drop-off locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts.

The data is stored in Parquet format (*.parquet*). The dataset is from 2009 to 2024. However, for this assignment, we will only be using the data from 2023.

The data for each month is present in a different parquet file. You will get twelve files for each of the months in 2023.

The data was collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers like vendors and taxi hailing apps. <br>

You can find the link to the TLC trip records page here: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

THE ABOVE LINK IS JUST FOR REFERENCE - PLEASE DO NOT USE THE DATASET PROVIDED IN THE ABOVE WEBPAGE FOR SOLVING THE ASSIGNMENT.

**Trip Records**



|Field Name       |description |
|:----------------|:-----------|
| VendorID | A code indicating the TPEP provider that provided the record. <br> 1= Creative Mobile Technologies, LLC; <br> 2= VeriFone Inc. |
| tpep_pickup_datetime | The date and time when the meter was engaged.  |
| tpep_dropoff_datetime | The date and time when the meter was disengaged.   |
| Passenger_count | The number of passengers in the vehicle. <br> This is a driver-entered value. |
| Trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged |
|RateCodeID |The final rate code in effect at the end of the trip.<br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br>4 = Nassau or Westchester <br>5 = Negotiated fare <br>6 = Group ride |
|Store_and_fwd_flag |This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server.  <br>Y= store and forward trip <br>N= not a store and forward trip |
|Payment_type| A numeric code signifying how the passenger paid for the trip. <br> 1 = Credit card <br>2 = Cash <br>3 = No charge <br>4 = Dispute <br>5 = Unknown <br>6 = Voided trip |
|Fare_amount| The time-and-distance fare calculated by the meter. <br>Extra Miscellaneous extras and surcharges.  Currently, this only includes the 0.50 and 1 USD rush hour and overnight charges. |
|MTA_tax |0.50 USD MTA tax that is automatically triggered based on the metered rate in use. |
|Improvement_surcharge | 0.30 USD improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
|Tip_amount |Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| Tolls_amount | Total amount of all tolls paid in trip.  |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
|Congestion_Surcharge |Total amount collected in trip for NYS congestion surcharge. |
| Airport_fee | 1.25 USD for pick up only at LaGuardia and John F. Kennedy Airports|

Although the amounts of extra charges and taxes applied are specified in the data dictionary, you will see that some cases have different values of these charges in the actual data.

**Taxi Zones**

Each of the trip records contains a field corresponding to the location of the pickup or drop-off of the trip, populated by numbers ranging from 1-263.

These numbers correspond to taxi zones, which may be downloaded as a table or map/shapefile and matched to the trip records using a join.

This is covered in more detail in later sections.

---

## **1** Data Preparation

<font color = red>[5 marks]</font> <br>

### Import Libraries

In [1]:
# Import warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Import the libraries you will be using for analysis
%pip install pandas matplotlib seaborn numpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# To read parquet files, we will need to install pyarrow or fastparquet.
%pip install fastparquet
import fastparquet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Recommended versions
# numpy version: 1.26.4
# pandas version: 2.2.2
# matplotlib version: 3.10.0
# seaborn version: 0.13.2

# Check versions
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)
print("fastparquet version:", fastparquet.__version__)

numpy version: 2.5.0
pandas version: 3.0.3
matplotlib version: 3.11.0
seaborn version: 0.13.2
fastparquet version: 2026.5.0


### **1.1** Load the dataset
<font color = red>[5 marks]</font> <br>

You will see twelve files, one for each month.

To read parquet files with Pandas, you have to follow a similar syntax as that for CSV files.

`df = pd.read_parquet('file.parquet')`

In [4]:
# Try loading one file

df = pd.read_parquet('DatasetsAndDictionary/trip_records/2023-1.parquet')
df.info()

<class 'pandas.DataFrame'>
Index: 3041714 entries, 0 to 3066765
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee            float64       


How many rows are there? Do you think handling such a large number of rows is computationally feasible when we have to combine the data for all twelve months into one?

To handle this, we need to sample a fraction of data from each of the files. How to go about that? Think of a way to select only some portion of the data from each month's file that accurately represents the trends.

#### Analyzing Data for Data Cleaning

In [5]:
print(df.shape)

(3041714, 19)


In [6]:
unique_years = df['tpep_pickup_datetime'].dt.year.sort_values().unique()
print(unique_years)

[2008 2022 2023]


In [8]:
data_for_2023 = df[df['tpep_pickup_datetime'].dt.year == 2023]
print(data_for_2023.shape)

(3041676, 19)


In [9]:
unique_dates = data_for_2023['tpep_pickup_datetime'].dt.strftime("%Y-%m-%d").sort_values().unique()
print(unique_dates)

<ArrowStringArray>
['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05',
 '2023-01-06', '2023-01-07', '2023-01-08', '2023-01-09', '2023-01-10',
 '2023-01-11', '2023-01-12', '2023-01-13', '2023-01-14', '2023-01-15',
 '2023-01-16', '2023-01-17', '2023-01-18', '2023-01-19', '2023-01-20',
 '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-01-25',
 '2023-01-26', '2023-01-27', '2023-01-28', '2023-01-29', '2023-01-30',
 '2023-01-31', '2023-02-01']
Length: 32, dtype: str


In [11]:
data_for_1_day = data_for_2023[data_for_2023['tpep_pickup_datetime'].dt.strftime("%Y-%m-%d") == unique_dates[0]]
hours_in_data = data_for_1_day['tpep_pickup_datetime'].dt.strftime("%H").sort_values().unique()
print(hours_in_data)

<ArrowStringArray>
['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12',
 '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23']
Length: 24, dtype: str


- Data files contains data for multiple year. However, we need to extract data only for year 2023. This would be handled while fetching all the data files.
- Data for different month is also found. Here we can see the data for month #2 (2023-02-01) in the data file for month #1 (2023-1.parquet). This would be automatically handled while fetching all the data files.
- The hours in `tpep_pickup_datetime` starts from '00' and ends at '23' which means the hours is in 24 hour format.

#### Sampling the Data
> One way is to take a small percentage of entries for pickup in every hour of a date. So, for all the days in a month, we can iterate through the hours and select 5% values randomly from those. Use `tpep_pickup_datetime` for this. Separate date and hour from the datetime values and then for each date, select some fraction of trips for each of the 24 hours.

To sample data, you can use the `sample()` method. Follow this syntax:

```Python
# sampled_data is an empty DF to keep appending sampled data of each hour
# hour_data is the DF of entries for an hour 'X' on a date 'Y'

sample = hour_data.sample(frac = 0.05, random_state = 42)
# sample 0.05 of the hour_data
# random_state is just a seed for sampling, you can define it yourself

sampled_data = pd.concat([sampled_data, sample]) # adding data for this hour to the DF
```

This *sampled_data* will contain 5% values selected at random from each hour.

Note that the code given above is only the part that will be used for sampling and not the complete code required for sampling and combining the data files.

Keep in mind that you sample by date AND hour, not just hour. (Why?)

##### Analyses for Sampling

- Since we have `3041676` entries for year 2023 in one parquet file.
- There would be around `3041676 * 12 = 36500112` entries across 12 parquet files
- If we sample around `5%` data also then we would have around `1825005` entries which would still be too much data.
- So we would be going with `1%` of sample data across hours for each day.
- So we would be having `36500112 * 0.01 = 365000` entries.

---

**1.1.1** <font color = red>[5 marks]</font> <br>
Figure out how to sample and combine the files.

**Note:** It is not mandatory to use the method specified above. While sampling, you only need to make sure that your sampled data represents the overall data of all the months accurately.

In [22]:
# Sample the data
# It is recommmended to not load all the files at once to avoid memory overload

In [23]:
# from google.colab import drive
# drive.mount('/content/drive')

In [21]:
# Take a small percentage of entries from each hour of every date.
# Iterating through the monthly data:
#   read a month file -> day -> hour: append sampled data -> move to next hour -> move to next day after 24 hours -> move to next month file
# Create a single dataframe for the year combining all the monthly data

# Select the folder having data files
import os

# Select the folder having data files
os.chdir('DatasetsAndDictionary/trip_records')
# os.chdir(os.getcwd() if os.path.basename(os.getcwd()) == 'trip_records' else os.path.abspath('DatasetsAndDictionary/trip_records'))

# Create a list of all the twelve files to read
file_list = os.listdir()

# initialise an empty dataframe
df = pd.DataFrame()

for file_name in file_list:
    try:
        # file path for the current file
        file_path = os.path.join(os.getcwd(), file_name)

        # Reading the current file
        current_df = pd.read_parquet(file_path)

        # Extract the data only for the year 2023
        data_for_2023 = current_df[current_df['tpep_pickup_datetime'].dt.year == 2023]

        # Loop through dates and then loop through every hour of each date
        for date in data_for_2023['tpep_pickup_datetime'].dt.strftime("%Y-%m-%d").unique():            
            date_df = data_for_2023[data_for_2023['tpep_pickup_datetime'].dt.strftime("%Y-%m-%d") == date]
            
            # We will store the sampled data for the current date in this df by appending the sampled data from each hour to this
            # After completing iteration through each date, we will append this data to the final dataframe.
            sampled_data = pd.DataFrame()

            # Iterate through each hour of the selected date
            hours = [f"{h:02d}" for h in range(24)]
            for hour in hours:
                hour_df = date_df[date_df['tpep_pickup_datetime'].dt.strftime("%H") == hour]
                
                # Sample 5% of the hourly data randomly
                sampled_hour_df = hour_df.sample(frac=0.01, random_state=10)
                
                # add data of this hour to the sampled_data dataframe
                sampled_data = pd.concat([sampled_data, sampled_hour_df], ignore_index=True)
            
            # Concatenate the sampled data of all the hours of current date to a single dataframe
            df = pd.concat([df, sampled_data], ignore_index=True)
            
    except Exception as e:
        print(f"Error reading file {file_name}: {e}")

    print(file_name, df.shape)
print("Total entries: ", df.shape)

2023-1.parquet (30416, 19)
2023-10.parquet (65270, 20)
2023-11.parquet (98302, 20)
2023-12.parquet (131632, 20)
2023-2.parquet (165354, 20)
2023-3.parquet (198099, 20)
2023-4.parquet (226035, 20)
2023-5.parquet (254921, 20)
2023-6.parquet (287505, 20)
2023-7.parquet (322321, 20)
2023-8.parquet (351080, 20)
2023-9.parquet (379268, 20)
Total entries:  (379268, 20)


After combining the data files into one DataFrame, convert the new DataFrame to a CSV or parquet file and store it to use directly.

Ideally, you can try keeping the total entries to around 250,000 to 300,000.

In [26]:
# Store the df in csv/parquet
df.to_parquet('2023_sampled.parquet')

## **2** Data Cleaning
<font color = red>[30 marks]</font> <br>

Now we can load the new data directly.

In [79]:
# Load the new data file
df = pd.read_parquet('2023_sampled.parquet')

In [80]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,Airport_fee
0,2,2023-01-01 00:45:18,2023-01-01 00:56:17,3.0,1.15,1.0,N,144,79,1,11.40,1.0,0.5,3.28,0.0,1.0,19.68,2.5,0.0,NaN
1,2,2023-01-01 00:49:00,2023-01-01 01:08:00,NaN,8.09,NaN,NaN,236,13,0,36.01,0.0,0.5,8.00,0.0,1.0,48.01,NaN,NaN,NaN
2,2,2023-01-01 00:14:50,2023-01-01 00:20:42,1.0,1.04,1.0,N,211,231,1,8.60,1.0,0.5,4.00,0.0,1.0,17.60,2.5,0.0,NaN
3,2,2023-01-01 00:33:10,2023-01-01 00:41:50,1.0,1.85,1.0,N,229,107,1,10.70,1.0,0.5,3.92,0.0,1.0,19.62,2.5,0.0,NaN
4,2,2023-01-01 00:11:25,2023-01-01 00:14:33,1.0,0.71,1.0,N,263,237,1,5.80,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,NaN


In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 379268 entries, 0 to 379267
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               379268 non-null  int64         
 1   tpep_pickup_datetime   379268 non-null  datetime64[us]
 2   tpep_dropoff_datetime  379268 non-null  datetime64[us]
 3   passenger_count        366796 non-null  float64       
 4   trip_distance          379268 non-null  float64       
 5   RatecodeID             366796 non-null  float64       
 6   store_and_fwd_flag     366796 non-null  str           
 7   PULocationID           379268 non-null  int64         
 8   DOLocationID           379268 non-null  int64         
 9   payment_type           379268 non-null  int64         
 10  fare_amount            379268 non-null  float64       
 11  extra                  379268 non-null  float64       
 12  mta_tax                379268 non-null  float64       


#### **2.1** Fixing Columns
<font color = red>[10 marks]</font> <br>

Fix/drop any columns as you seem necessary in the below sections

**2.1.1** <font color = red>[2 marks]</font> <br>

Fix the index and drop unnecessary columns

In [82]:
# Fix the index and drop any columns that are not needed
df.reset_index(drop=True, inplace=True)


In [83]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,Airport_fee
0,2,2023-01-01 00:45:18,2023-01-01 00:56:17,3.0,1.15,1.0,N,144,79,1,11.40,1.0,0.5,3.28,0.0,1.0,19.68,2.5,0.0,NaN
1,2,2023-01-01 00:49:00,2023-01-01 01:08:00,NaN,8.09,NaN,NaN,236,13,0,36.01,0.0,0.5,8.00,0.0,1.0,48.01,NaN,NaN,NaN
2,2,2023-01-01 00:14:50,2023-01-01 00:20:42,1.0,1.04,1.0,N,211,231,1,8.60,1.0,0.5,4.00,0.0,1.0,17.60,2.5,0.0,NaN
3,2,2023-01-01 00:33:10,2023-01-01 00:41:50,1.0,1.85,1.0,N,229,107,1,10.70,1.0,0.5,3.92,0.0,1.0,19.62,2.5,0.0,NaN
4,2,2023-01-01 00:11:25,2023-01-01 00:14:33,1.0,0.71,1.0,N,263,237,1,5.80,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,NaN


**2.1.2** <font color = red>[3 marks]</font> <br>
There are two airport fee columns. This is possibly an error in naming columns. Let's see whether these can be combined into a single column.

In [84]:
# Combine the two airport fee columns
df["airport_fee_merged"] = df[["airport_fee", "Airport_fee"]].bfill(axis=1).iloc[:, 0]

In [85]:
# Before dropping the columns, let's check if there are any rows where both columns have values and they differ. If there are such rows, we need to investigate further before dropping the columns.
df[~df['airport_fee'].isnull() & ~df['Airport_fee'].isnull() & (df['airport_fee'] != df['Airport_fee'])]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,Airport_fee,airport_fee_merged


In [86]:
# Since there are no rows where both columns have values and they differ, we can safely drop the original columns.
df.drop(columns=['airport_fee', 'Airport_fee'], inplace=True)
df.rename(columns={"airport_fee_merged": "airport_fee"}, inplace=True)

**2.1.3** <font color = red>[5 marks]</font> <br>
Fix columns with negative (monetary) values

In [88]:
# check where values of fare amount are negative
negative_fare_rows = df[df['fare_amount'] < 0.0]
negative_fare_rows

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee


- There is no row with negative fare amount

Did you notice something different in the `RatecodeID` column for above records?

In [89]:
# Analyse RatecodeID for the negative fare amounts
df['RatecodeID'].info()
print(df['RatecodeID'].unique())

<class 'pandas.Series'>
RangeIndex: 379268 entries, 0 to 379267
Series name: RatecodeID
Non-Null Count   Dtype  
--------------   -----  
366796 non-null  float64
dtypes: float64(1)
memory usage: 2.9 MB
[ 1. nan  4.  5.  2. 99.  3.]


- RatecodeID has a datatype of `float64` but the acceptable values are `1, 2, 3, 4, 5 and 6`. So we need to convert it into `int64`
- There are also some rows where the RatecodeID is `99.`. Since this is not an acceptable value, we need to handle this.

In [91]:
df["RatecodeID"] = df["RatecodeID"].astype("Int64")

In [92]:
df['RatecodeID'].unique()

<IntegerArray>
[1, <NA>, 4, 5, 2, 99, 3]
Length: 7, dtype: Int64

In [122]:
null_ratecode = df[df['RatecodeID'].isna()]

print(len(null_ratecode))
print(len(null_ratecode) / len(df) * 100)


12472
3.288439836738138


In [123]:
null_ratecode.head(20)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
1,2,2023-01-01 00:49:00,2023-01-01 01:08:00,NaN,8.09,<NA>,NaN,236,13,0,36.01,0.0,0.5,8.00,0.00,1.0,48.01,NaN,NaN
12,2,2023-01-01 00:45:00,2023-01-01 01:09:00,NaN,9.66,<NA>,NaN,238,33,0,37.45,0.0,0.5,8.29,0.00,1.0,49.74,NaN,NaN
24,2,2023-01-01 00:14:00,2023-01-01 00:24:00,NaN,0.00,<NA>,NaN,170,79,0,13.19,0.0,0.5,3.44,0.00,1.0,20.63,NaN,NaN
41,2,2023-01-01 00:13:40,2023-01-01 00:33:06,NaN,2.79,<NA>,NaN,79,161,0,19.48,0.0,0.5,3.52,0.00,1.0,27.00,NaN,NaN
44,2,2023-01-01 00:56:37,2023-01-01 01:28:08,NaN,5.49,<NA>,NaN,36,25,0,27.18,0.0,0.5,5.74,0.00,1.0,34.42,NaN,NaN
54,2,2023-01-01 01:16:19,2023-01-01 01:23:19,NaN,2.26,<NA>,NaN,141,74,0,13.85,0.0,0.5,3.57,0.00,1.0,21.42,NaN,NaN
76,1,2023-01-01 01:52:39,2023-01-01 02:00:05,NaN,1.30,<NA>,NaN,236,239,0,8.60,1.0,0.5,2.04,0.00,1.0,15.64,NaN,NaN
79,2,2023-01-01 01:49:00,2023-01-01 02:16:00,NaN,7.92,<NA>,NaN,36,237,0,38.99,0.0,0.5,4.30,0.00,1.0,47.29,NaN,NaN
82,2,2023-01-01 01:14:00,2023-01-01 01:40:00,NaN,7.00,<NA>,NaN,151,148,0,36.89,0.0,0.5,8.18,0.00,1.0,49.07,NaN,NaN
89,2,2023-01-01 01:20:38,2023-01-01 01:30:58,NaN,1.75,<NA>,NaN,75,262,0,13.63,0.0,0.5,2.64,0.00,1.0,20.27,NaN,NaN


In [124]:
ratecode99 = df[df['RatecodeID'] == 99]

print(len(ratecode99))
print(len(ratecode99) / len(df) * 100)


2098
0.5531708448906842


In [125]:
ratecode99.head(20)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
141,1,2023-01-01 02:54:18,2023-01-01 03:14:41,1.0,3.7,99,N,77,89,1,17.2,0.0,0.5,0.0,0.00,1.0,18.70,0.0,0.0
260,1,2023-01-01 08:44:33,2023-01-01 09:19:54,1.0,2.5,99,N,225,35,1,19.5,0.0,0.5,0.0,0.00,1.0,21.00,0.0,0.0
353,1,2023-01-01 12:40:14,2023-01-01 13:15:58,1.0,0.0,99,N,77,188,1,21.0,0.0,0.5,0.0,0.00,1.0,22.50,0.0,0.0
385,1,2023-01-01 13:12:21,2023-01-01 14:21:03,1.0,0.0,99,N,223,143,1,26.5,0.0,0.5,0.0,6.55,1.0,34.55,0.0,0.0
635,1,2023-01-01 19:20:53,2023-01-01 19:52:05,1.0,4.7,99,N,69,3,1,37.5,0.0,0.5,0.0,0.00,1.0,39.00,0.0,0.0
809,1,2023-01-02 07:45:19,2023-01-02 08:03:57,1.0,1.8,99,N,56,95,1,18.5,0.0,0.5,0.0,0.00,1.0,20.00,0.0,0.0
918,1,2023-01-02 11:07:02,2023-01-02 11:53:54,1.0,11.4,99,N,117,130,1,37.5,0.0,0.5,0.0,0.00,1.0,39.00,0.0,0.0
1458,1,2023-01-03 06:23:30,2023-01-03 06:34:40,1.0,2.1,99,N,153,174,1,20.5,0.0,0.5,0.0,0.00,1.0,22.00,0.0,0.0
1514,1,2023-01-03 08:21:19,2023-01-03 08:50:36,1.0,4.2,99,N,241,247,1,25.5,0.0,0.5,0.0,0.00,1.0,27.00,0.0,0.0
1537,1,2023-01-03 08:24:12,2023-01-03 08:33:33,1.0,0.5,99,N,247,235,1,16.5,0.0,0.5,0.0,0.00,1.0,18.00,0.0,0.0


- RatecodeID is null: A total of 12,472 records (3.29%) have a missing RatecodeID. This indicates a notable subset of records with incomplete metadata and warrants further investigation. However, these records should not be removed solely based on this field, as they still represent valid trips with meaningful values for distance, fare, total amount, and pickup/dropoff information. The approach would to assess the completeness and validity of the core trip attributes before deciding whether to retain them as incomplete records or exclude them from downstream analysis.

- RatecodeID = 99: A total of 2,098 records (0.55%) have RatecodeID = 99. While the original dataset description lists valid rate codes from 1 to 6, documentation could identify 99 as a "Null/Unknown" rate code rather than an invalid value. Therefore, these records should not be automatically discarded. Given their small proportion in the dataset, it is reasonable to retain them as an**"Unknown Rate Code"** category and exclude them only from analyses that specifically require valid rate-code classifications.



In [128]:
# Find which columns have negative values
# Check all the quantitve columns for negative values
negative_passenger_count_rows = df[df['passenger_count'] < 0.0]
print("Negative passenger count rows: ", negative_passenger_count_rows.shape[0])
negative_trip_distance_rows = df[df['trip_distance'] < 0.0]
print("Negative trip distance rows: ", negative_trip_distance_rows.shape[0])
negative_fare_amount_rows = df[df['fare_amount'] < 0.0]
print("Negative fare amount rows: ", negative_fare_amount_rows.shape[0])
negative_extra_rows = df[df['extra'] < 0.0]
print("Negative extra rows: ", negative_extra_rows.shape[0])
negative_mta_tax_rows = df[df['mta_tax'] < 0.0]
print("Negative mta tax rows: ", negative_mta_tax_rows.shape[0])
negative_tip_amount_rows = df[df['tip_amount'] < 0.0]
print("Negative tip amount rows: ", negative_tip_amount_rows.shape[0])
negative_tolls_amount_rows = df[df['tolls_amount'] < 0.0]
print("Negative tolls amount rows: ", negative_tolls_amount_rows.shape[0])
negative_improvement_surcharge_rows = df[df['improvement_surcharge'] < 0.0]
print("Negative improvement surcharge rows: ", negative_improvement_surcharge_rows.shape[0])
negative_total_amount_rows = df[df['total_amount'] < 0.0]
print("Negative total amount rows: ", negative_total_amount_rows.shape[0])
negative_congestion_surcharge_rows = df[df['congestion_surcharge'] < 0.0]
print("Negative congestion surcharge rows: ", negative_congestion_surcharge_rows.shape[0])
negative_airport_fee_rows = df[df['airport_fee'] < 0.0]
print("Negative airport fee rows: ", negative_airport_fee_rows.shape[0])

Negative passenger count rows:  0
Negative trip distance rows:  0
Negative fare amount rows:  0
Negative extra rows:  0
Negative mta tax rows:  18
Negative tip_amount rows:  0
Negative tolls_amount rows:  0
Negative improvement surcharge rows:  20
Negative total amount rows:  20
Negative congestion surcharge rows:  15
Negative airport fee rows:  6


- We see that we have negative values for following columns:
1. `mta_tax`: 18
2. `improvement_surcharge`: 20
3. `total_amount`: 20
4. `congestion_surcharge`: 15
5. `airport_fee`: 6

In [132]:
df_with_negative_quantative_columns = df[
    (df['mta_tax'] < 0.0)
    | (df['improvement_surcharge'] < 0.0)
    | (df['total_amount'] < 0.0)
    | (df['congestion_surcharge'] < 0.0)
    | (df['airport_fee'] < 0.0)
]
df_with_negative_quantative_columns.head(20)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
20128,2,2023-01-21 18:24:52,2023-01-21 18:35:40,1.0,1.27,1,N,263,140,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-4.00,-2.5,0.00
68344,2,2023-11-03 15:51:42,2023-11-03 15:53:00,1.0,0.21,2,N,246,246,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-4.00,-2.5,0.00
76507,2,2023-11-10 15:43:00,2023-11-10 15:43:06,1.0,0.00,2,N,138,138,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-3.25,0.0,-1.75
102445,2,2023-12-04 17:30:33,2023-12-04 17:32:03,1.0,0.22,1,N,161,164,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-4.00,-2.5,0.00
113987,2,2023-12-13 22:22:52,2023-12-13 22:55:49,2.0,21.49,1,N,68,265,2,0.0,0.0,0.0,0.0,0.0,-1.0,-1.00,0.0,0.00
142255,2,2023-03-10 16:18:09,2023-03-10 16:49:43,3.0,6.94,1,N,88,230,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-4.00,-2.5,0.00
146189,2,2023-03-14 12:44:48,2023-03-14 13:03:02,1.0,1.57,1,N,231,148,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-4.00,-2.5,0.00
167174,2,2023-06-02 16:53:58,2023-06-02 17:08:38,1.0,1.58,1,N,43,143,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-4.00,-2.5,0.00
179289,2,2023-06-13 13:20:04,2023-06-13 13:33:04,3.0,1.92,1,N,48,238,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-4.00,-2.5,0.00
196477,2,2023-06-29 12:37:19,2023-06-29 13:11:30,1.0,9.00,1,N,138,161,2,0.0,0.0,-0.5,0.0,0.0,-1.0,-5.75,-2.5,-1.75


- There are 20 records where one or more monetary charge columns have negative values. All of these records have payment_type as Cash and fare_amount = 0, which is inconsistent with a normal completed taxi trip. These records likely represent correction, reversal, voided, or vendor-side data-quality anomalies rather than valid passenger fare transactions. Since the count is extremely small, they can be safely removed from the cleaned dataset, especially for revenue, fare, and surcharge-related analysis.

In [ ]:
# fix these negative values
df = df.drop(df_with_negative_quantative_columns.index)

### **2.2** Handling Missing Values
<font color = red>[10 marks]</font> <br>

**2.2.1**  <font color = red>[2 marks]</font> <br>
Find the proportion of missing values in each column




In [137]:
# Find the proportion of missing values in each column
df.isnull().sum()

VendorID                     0
tpep_pickup_datetime         0
tpep_dropoff_datetime        0
passenger_count          12472
trip_distance                0
RatecodeID               12472
store_and_fwd_flag       12472
PULocationID                 0
DOLocationID                 0
payment_type                 0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
improvement_surcharge        0
total_amount                 0
congestion_surcharge     12472
airport_fee              12472
dtype: int64

In [176]:
# Confirm these are the same records
print((
    df["RatecodeID"].isna() &
    df["passenger_count"].isna() &
    df["store_and_fwd_flag"].isna() &
    df["congestion_surcharge"].isna() &
    df["airport_fee"].isna()
).sum())

12472


- Missing values are concentrated in five columns: passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge, and airport_fee, each containing exactly 12,472 null values (3.29% of the dataset). The identical null counts suggest that these fields belong to a specific subset of records rather than representing independent missing-data issues. Since these records also commonly exhibit payment_type = 0, they likely correspond to a special trip category or incomplete vendor submissions. Therefore, the records should not be removed solely due to missing values; instead, they should be flagged and handled separately depending on the analysis being performed.

**2.2.2**  <font color = red>[3 marks]</font> <br>
Handling missing values in `passenger_count`

In [179]:
# Display the rows with null values
# Impute NaN values in 'passenger_count'
df['passenger_count'].isna().sum()

np.int64(12472)

In [197]:
invalid_records = df[
    ((df['passenger_count'].isna())
    | (df['passenger_count'] == 0.0))
    & (df['fare_amount'] == 0.0)
     & (df['total_amount'] == 0.0)
]
invalid_records.shape

(3, 19)

- Identified 3 records where passenger_count is either missing or zero, and both fare_amount and total_amount are zero. These records do not represent meaningful completed taxi trips because they lack both passenger information and monetary value. Given the extremely small count and limited analytical usefulness, these records are treated as invalid or incomplete trip records and removed from the cleaned dataset.

In [198]:
df = df.drop(invalid_records.index)

Did you find zeroes in passenger_count? Handle these.

In [203]:
zero_passenger_count = df[
    (df['passenger_count'] == 0.0)
]
zero_passenger_count.shape

(5841, 19)

**2.2.3**  <font color = red>[2 marks]</font> <br>
Handle missing values in `RatecodeID`

In [ ]:
# Fix missing values in 'RatecodeID'


**2.2.4**  <font color = red>[3 marks]</font> <br>
Impute NaN in `congestion_surcharge`

In [ ]:
# handle null values in congestion_surcharge




Are there missing values in other columns? Did you find NaN values in some other set of columns? Handle those missing values below.

In [ ]:
# Handle any remaining missing values



### **2.3** Handling Outliers
<font color = red>[10 marks]</font> <br>

Before we start fixing outliers, let's perform outlier analysis.

In [ ]:
# Describe the data and check if there are any potential outliers present
# Check for potential out of place values in various columns



**2.3.1**  <font color = red>[10 marks]</font> <br>
Based on the above analysis, it seems that some of the outliers are present due to errors in registering the trips. Fix the outliers.

Some points you can look for:
- Entries where `trip_distance` is nearly 0 and `fare_amount` is more than 300
- Entries where `trip_distance` and `fare_amount` are 0 but the pickup and dropoff zones are different (both distance and fare should not be zero for different zones)
- Entries where `trip_distance` is more than 250  miles.
- Entries where `payment_type` is 0 (there is no payment_type 0 defined in the data dictionary)

These are just some suggestions. You can handle outliers in any way you wish, using the insights from above outlier analysis.

How will you fix each of these values? Which ones will you drop and which ones will you replace?

First, let us remove 7+ passenger counts as there are very less instances.

In [ ]:
# remove passenger_count > 6


In [ ]:
# Continue with outlier handling



In [ ]:
# Do any columns need standardising?



## **3** Exploratory Data Analysis
<font color = red>[90 marks]</font> <br>

In [ ]:
df.columns.tolist()

#### **3.1** General EDA: Finding Patterns and Trends
<font color = red>[40 marks]</font> <br>

**3.1.1** <font color = red>[3 marks]</font> <br>
Categorise the varaibles into Numerical or Categorical.
* `VendorID`:
* `tpep_pickup_datetime`:
* `tpep_dropoff_datetime`:
* `passenger_count`:
* `trip_distance`:
* `RatecodeID`:
* `PULocationID`:
* `DOLocationID`:
* `payment_type`:
* `pickup_hour`:
* `trip_duration`:


The following monetary parameters belong in the same category, is it categorical or numerical?


* `fare_amount`
* `extra`
* `mta_tax`
* `tip_amount`
* `tolls_amount`
* `improvement_surcharge`
* `total_amount`
* `congestion_surcharge`
* `airport_fee`

##### Temporal Analysis

**3.1.2** <font color = red>[5 marks]</font> <br>
Analyse the distribution of taxi pickups by hours, days of the week, and months.

In [ ]:
# Find and show the hourly trends in taxi pickups



In [ ]:
# Find and show the daily trends in taxi pickups (days of the week)



In [ ]:
# Show the monthly trends in pickups



##### Financial Analysis

Take a look at the financial parameters like `fare_amount`, `tip_amount`, `total_amount`, and also `trip_distance`. Do these contain zero/negative values?

In [ ]:
# Analyse the above parameters



Do you think it is beneficial to create a copy DataFrame leaving out the zero values from these?

**3.1.3** <font color = red>[2 marks]</font> <br>
Filter out the zero values from the above columns.

**Note:** The distance might be 0 in cases where pickup and drop is in the same zone. Do you think it is suitable to drop such cases of zero distance?

In [ ]:
# Create a df with non zero entries for the selected parameters.



**3.1.4** <font color = red>[3 marks]</font> <br>
Analyse the monthly revenue (`total_amount`) trend

In [ ]:
# Group data by month and analyse monthly revenue



**3.1.5** <font color = red>[3 marks]</font> <br>
Show the proportion of each quarter of the year in the revenue

In [ ]:
# Calculate proportion of each quarter



**3.1.6** <font color = red>[3 marks]</font> <br>
Visualise the relationship between `trip_distance` and `fare_amount`. Also find the correlation value for these two.

**Hint:** You can leave out the trips with trip_distance = 0

In [ ]:
# Show how trip fare is affected by distance



**3.1.7** <font color = red>[5 marks]</font> <br>
Find and visualise the correlation between:
1. `fare_amount` and trip duration (pickup time to dropoff time)
2. `fare_amount` and `passenger_count`
3. `tip_amount` and `trip_distance`

In [ ]:
# Show relationship between fare and trip duration



In [ ]:
# Show relationship between fare and number of passengers



In [ ]:
# Show relationship between tip and trip distance



**3.1.8** <font color = red>[3 marks]</font> <br>
Analyse the distribution of different payment types (`payment_type`)

In [ ]:
# Analyse the distribution of different payment types (payment_type).




- 1= Credit card
- 2= Cash
- 3= No charge
- 4= Dispute



##### Geographical Analysis

For this, you have to use the *taxi_zones.shp* file from the *taxi_zones* folder.

There would be multiple files inside the folder (such as *.shx, .sbx, .sbn* etc). You do not need to import/read any of the files other than the shapefile, *taxi_zones.shp*.

Do not change any folder structure - all the files need to be present inside the folder for it to work.

The folder structure should look like this:
```
Taxi Zones
|- taxi_zones.shp.xml
|- taxi_zones.prj
|- taxi_zones.sbn
|- taxi_zones.shp
|- taxi_zones.dbf
|- taxi_zones.shx
|- taxi_zones.sbx

 ```

 You only need to read the `taxi_zones.shp` file. The *shp* file will utilise the other files by itself.

We will use the *GeoPandas* library for geopgraphical analysis
```
import geopandas as gpd
```

More about geopandas and shapefiles: [About](https://geopandas.org/en/stable/about.html)


Reading the shapefile is very similar to *Pandas*. Use `gpd.read_file()` function to load the data (*taxi_zones.shp*) as a GeoDataFrame. Documentation: [Reading and Writing Files](https://geopandas.org/en/stable/docs/user_guide/io.html)

In [ ]:
# !pip install geopandas

**3.1.9** <font color = red>[2 marks]</font> <br>
Load the shapefile and display it.

In [ ]:
# import geopandas as gpd


# Read the shapefile using geopandas
zones = # read the .shp file using gpd
zones.head()

Now, if you look at the DataFrame created, you will see columns like: `OBJECTID`,`Shape_Leng`, `Shape_Area`, `zone`, `LocationID`, `borough`, `geometry`.
<br><br>

Now, the `locationID` here is also what we are using to mark pickup and drop zones in the trip records.

The geometric parameters like shape length, shape area and geometry are used to plot the zones on a map.

This can be easily done using the `plot()` method.

In [ ]:
# print(zones.info())
# zones.plot()

Now, you have to merge the trip records and zones data using the location IDs.



**3.1.10** <font color = red>[3 marks]</font> <br>
Merge the zones data into trip data using the `locationID` and `PULocationID` columns.

In [ ]:
# Merge zones and trip records using locationID and PULocationID



**3.1.11** <font color = red>[3 marks]</font> <br>
Group data by location IDs to find the total number of trips per location ID

In [ ]:
# Group data by location and calculate the number of trips



**3.1.12** <font color = red>[2 marks]</font> <br>
Now, use the grouped data to add number of trips to the GeoDataFrame.

We will use this to plot a map of zones showing total trips per zone.

In [ ]:
# Merge trip counts back to the zones GeoDataFrame




The next step is creating a color map (choropleth map) showing zones by the number of trips taken.

Again, you can use the `zones.plot()` method for this. [Plot Method GPD](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html#geopandas.GeoDataFrame.plot)

But first, you need to define the figure and axis for the plot.

`fig, ax = plt.subplots(1, 1, figsize = (12, 10))`

This function creates a figure (fig) and a single subplot (ax)

---

After setting up the figure and axis, we can proceed to plot the GeoDataFrame on this axis. This is done in the next step where we use the plot method of the GeoDataFrame.

You can define the following parameters in the `zones.plot()` method:
```
column = '',
ax = ax,
legend = True,
legend_kwds = {'label': "label", 'orientation': "<horizontal/vertical>"}
```

To display the plot, use `plt.show()`.

**3.1.13** <font color = red>[3 marks]</font> <br>
Plot a color-coded map showing zone-wise trips

In [ ]:
# Define figure and axis


# Plot the map and display it



In [ ]:
# can you try displaying the zones DF sorted by the number of trips?



Here we have completed the temporal, financial and geographical analysis on the trip records.

**Compile your findings from general analysis below:**

You can consider the following points:

* Busiest hours, days and months
* Trends in revenue collected
* Trends in quarterly revenue
* How fare depends on trip distance, trip duration and passenger counts
* How tip amount depends on trip distance
* Busiest zones


#### **3.2** Detailed EDA: Insights and Strategies
<font color = red>[50 marks]</font> <br>

Having performed basic analyses for finding trends and patterns, we will now move on to some detailed analysis focussed on operational efficiency, pricing strategies, and customer experience.

##### Operational Efficiency

Analyze variations by time of day and location to identify bottlenecks or inefficiencies in routes

**3.2.1** <font color = red>[3 marks]</font> <br>
Identify slow routes by calculating the average time taken by cabs to get from one zone to another at different hours of the day.

Speed on a route *X* for hour *Y* = (*distance of the route X / average trip duration for hour Y*)

In [ ]:
# Find routes which have the slowest speeds at different times of the day



How does identifying high-traffic, high-demand routes help us?

**3.2.2** <font color = red>[3 marks]</font> <br>
Calculate the number of trips at each hour of the day and visualise them. Find the busiest hour and show the number of trips for that hour.

In [ ]:
# Visualise the number of trips per hour and find the busiest hour



Remember, we took a fraction of trips. To find the actual number, you have to scale the number up by the sampling ratio.

**3.2.3** <font color = red>[2 mark]</font> <br>
Find the actual number of trips in the five busiest hours

In [ ]:
# Scale up the number of trips

# Fill in the value of your sampling fraction and use that to scale up the numbers
sample_fraction =



**3.2.4** <font color = red>[3 marks]</font> <br>
Compare hourly traffic pattern on weekdays. Also compare for weekend.

In [ ]:
# Compare traffic trends for the week days and weekends



What can you infer from the above patterns? How will finding busy and quiet hours for each day help us?

**3.2.5** <font color = red>[3 marks]</font> <br>
Identify top 10 zones with high hourly pickups. Do the same for hourly dropoffs. Show pickup and dropoff trends in these zones.

In [ ]:
# Find top 10 pickup and dropoff zones



**3.2.6** <font color = red>[3 marks]</font> <br>
Find the ratio of pickups and dropoffs in each zone. Display the 10 highest (pickup/drop) and 10 lowest (pickup/drop) ratios.

In [ ]:
# Find the top 10 and bottom 10 pickup/dropoff ratios



**3.2.7** <font color = red>[3 marks]</font> <br>
Identify zones with high pickup and dropoff traffic during night hours (11PM to 5AM)

In [ ]:
# During night hours (11pm to 5am) find the top 10 pickup and dropoff zones
# Note that the top zones should be of night hours and not the overall top zones



Now, let us find the revenue share for the night time hours and the day time hours. After this, we will move to deciding a pricing strategy.

**3.2.8** <font color = red>[2 marks]</font> <br>
Find the revenue share for nighttime and daytime hours.

In [ ]:
# Filter for night hours (11 PM to 5 AM)



##### Pricing Strategy

**3.2.9** <font color = red>[2 marks]</font> <br>
For the different passenger counts, find the average fare per mile per passenger.

For instance, suppose the average fare per mile for trips with 3 passengers is 3 USD/mile, then the fare per mile per passenger will be 1 USD/mile.

In [ ]:
# Analyse the fare per mile per passenger for different passenger counts




**3.2.10** <font color = red>[3 marks]</font> <br>
Find the average fare per mile by hours of the day and by days of the week

In [ ]:
# Compare the average fare per mile for different days and for different times of the day



**3.2.11** <font color = red>[3 marks]</font> <br>
Analyse the average fare per mile for the different vendors for different hours of the day

In [ ]:
# Compare fare per mile for different vendors



**3.2.12** <font color = red>[5 marks]</font> <br>
Compare the fare rates of the different vendors in a tiered fashion. Analyse the average fare per mile for distances upto 2 miles. Analyse the fare per mile for distances from 2 to 5 miles. And then for distances more than 5 miles.


In [ ]:
# Defining distance tiers



##### Customer Experience and Other Factors

**3.2.13** <font color = red>[5 marks]</font> <br>
Analyse average tip percentages based on trip distances, passenger counts and time of pickup. What factors lead to low tip percentages?

In [ ]:
#  Analyze tip percentages based on distances, passenger counts and pickup times



Additional analysis [optional]: Let's try comparing cases of low tips with cases of high tips to find out if we find a clear aspect that drives up the tipping behaviours

In [ ]:
# Compare trips with tip percentage < 10% to trips with tip percentage > 25%



**3.2.14** <font color = red>[3 marks]</font> <br>
Analyse the variation of passenger count across hours and days of the week.

In [ ]:
# See how passenger count varies across hours and days




**3.2.15** <font color = red>[2 marks]</font> <br>
Analyse the variation of passenger counts across zones

In [ ]:
# How does passenger count vary across zones



In [ ]:
# For a more detailed analysis, we can use the zones_with_trips GeoDataFrame
# Create a new column for the average passenger count in each zone.



Find out how often surcharges/extra charges are applied to understand their prevalance

**3.2.16** <font color = red>[5 marks]</font> <br>
Analyse the pickup/dropoff zones or times when extra charges are applied more frequently

In [ ]:
# How often is each surcharge applied?



## **4** Conclusion
<font color = red>[15 marks]</font> <br>

### **4.1** Final Insights and Recommendations
<font color = red>[15 marks]</font> <br>

Conclude your analyses here. Include all the outcomes you found based on the analysis.

Based on the insights, frame a concluding story explaining suitable parameters such as location, time of the day, day of the week etc. to be kept in mind while devising a strategy to meet customer demand and optimise supply.

**4.1.1** <font color = red>[5 marks]</font> <br>
Recommendations to optimize routing and dispatching based on demand patterns and operational inefficiencies

**4.1.2** <font color = red>[5 marks]</font> <br>

Suggestions on strategically positioning cabs across different zones to make best use of insights uncovered by analysing trip trends across time, days and months.

**4.1.3** <font color = red>[5 marks]</font> <br>
Propose data-driven adjustments to the pricing strategy to maximize revenue while maintaining competitive rates with other vendors.